In [0]:
%pip install langchain-community databricks-langchain databricks-vector-search databricks-sdk

In [0]:
import os
from langchain_community.document_loaders import WebBaseLoader
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
from databricks.vector_search.client import VectorSearchClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
import warnings
from databricks.sdk.runtime import dbutils
import time

In [0]:
table_name = "dsa.silver.dsa_documents"
index_name = "dsa.model.dsa_documents_index"
endpoint_name = "victor_salsicha"

In [0]:
os.environ["DATABRICKS_HOST"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [0]:
dsa_loader = WebBaseLoader("https://blog.dsacademy.com.br/microsoft-fabric-transformando-dados-em-conhecimento/")
documentos = dsa_loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1800, chunk_overlap = 200)
docs = text_splitter.split_documents(documentos)

In [0]:
docs_data = [
    {
        "doc_id": str(i),
        "content": doc.page_content,
        "source": doc.metadata.get("source", "unknown")
    }
    for i, doc in enumerate(docs)
]

df = spark.createDataFrame(docs_data)

df.write.format("delta").mode("overwrite").saveAsTable(table_name)
spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:
vs_client = VectorSearchClient(disable_notice=True)

In [0]:
endpoint_exists = False

print(f"Fetching information for endpoint: '{endpoint_name}'...")

# 1. CHECK IF IT EXISTS FIRST
try:
    # If the endpoint already exists, this call returns its data
    # If it does NOT exist, the API raises an exception (RESOURCE_DOES_NOT_EXIST)
    endpoint_info = vs_client.get_endpoint(name=endpoint_name)
    print(f"✅ The endpoint '{endpoint_name}' already exists! Skipping creation.")
    endpoint_exists = True

except Exception as e:
    # Capture the error and check if it's the "Not Found" error
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"⚠️ The endpoint '{endpoint_name}' was not found.")
        endpoint_exists = False
    else:
        # If it's a permission or network error, stop the program
        print(f"Unexpected error while fetching the endpoint: {e}")
        raise e

# 2. CREATE ONLY IF IT DOES NOT EXIST
if not endpoint_exists:
    print(f"Starting creation of endpoint '{endpoint_name}'...")
    try:
        vs_client.create_endpoint(
            name=endpoint_name,
            endpoint_type="STANDARD" # or "STORAGE_OPTIMIZED"
        )
        print("Creation command sent successfully!")
    except Exception as create_e:
        print(f"Error while trying to create the endpoint: {create_e}")
        raise create_e

# ==========================================
# 3. Wait Loop (Waiting for ONLINE status)
# ==========================================
print(f"Waiting for endpoint '{endpoint_name}' to become ONLINE to receive indexes...")

while True:
    try:
        # Fetch updated information about the endpoint from the server
        endpoint_info = vs_client.get_endpoint(name=endpoint_name)
        
        # The status is inside endpoint_status > state
        status = endpoint_info.get("endpoint_status", {}).get("state")
        
        if status == "ONLINE":
            print(f"🚀 Endpoint '{endpoint_name}' is ONLINE and ready to use!")
            break
        elif status == "PROVISIONING":
            print(f"⏳ Endpoint is provisioning machines... Status: {status}")
        else:
            # It may be OFFLINE or in another state
            print(f"⏳ Current status: {status}. Waiting...")
            
    except Exception as wait_e:
        print(f"Error while checking status during wait: {wait_e}")
        # Depending on your tolerance, you can raise wait_e or just continue the loop
        
    time.sleep(30) # Sleep 30 seconds to avoid flooding the API

In [0]:
index_exists = False
index = None

print(f"Checking status for index '{index_name}' on endpoint '{endpoint_name}'...")

# 1. EXPLICIT EXISTENCE CHECK
try:
    # If the index exists, this returns a VectorSearchIndex object
    # If it does NOT exist, the API raises a RESOURCE_DOES_NOT_EXIST exception
    index = vs_client.get_index(endpoint_name=endpoint_name, index_name=index_name)
    print(f"✅ Index '{index_name}' already exists. Skipping creation.")
    index_exists = True

except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"⚠️ Index '{index_name}' not found. It will be created.")
        index_exists = False
    else:
        # Halt execution on unexpected errors (e.g. permission denied)
        print(f"Unexpected error while fetching index status: {e}")
        raise e

# 2. CREATE IF MISSING
if not index_exists:
    print(f"Initiating creation for delta sync index '{index_name}'...")
    try:
        index = vs_client.create_delta_sync_index(
            endpoint_name=endpoint_name,
            source_table_name=table_name,
            index_name=index_name,
            pipeline_type="TRIGGERED",
            primary_key="doc_id",
            embedding_source_column="content",
            embedding_model_endpoint_name="databricks-gte-large-en"
        )
        print("Creation command sent successfully!")
    except Exception as create_e:
        print(f"Failed to create index: {create_e}")
        raise create_e

# ==========================================
# 3. READINESS POLLING LOOP
# ==========================================
print(f"Waiting for index '{index_name}' to be ready for queries...")

while True:
    try:
        # CRITICAL: We must use .describe() to retrieve the properties dictionary
        # Calling .get() directly on the object will raise an AttributeError!
        index_info = index.describe()
        status_dict = index_info.get("status", {})
        
        is_ready = status_dict.get("ready", False)
        message = status_dict.get("message", "Processing...")
        
        if is_ready:
            print(f"🚀 Index '{index_name}' is READY and fully synced!")
            break
        else:
            print(f"⏳ Syncing data... Status message: {message}")
            
    except Exception as wait_e:
        print(f"Error checking status during wait loop: {wait_e}")
        
    time.sleep(30) # Sleep for 30 seconds to respect API rate limits


In [0]:
dsa_vector_db = DatabricksVectorSearch(
    index_name=index_name
)

retriever = dsa_vector_db.as_retriever(search_kwargs={"k": 4})

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=200)

PROMPT_TEMPLATE = """
        Humano: Você é um assistente de IA e fornece respostas a perguntas do usuário.

        Use as seguintes informações para fornecer uma resposta concisa à pergunta entre as tags <question>.
        Se você não sabe a resposta, apenas diga que não sabe, não tente inventar uma resposta.

        <context>
        {context}
        </context>

        <question>
        {question}
        </question>

        A resposta deve ser específica e usar apenas informações confiáveis.

        Assistente:"""

prompt = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=["context", "question"])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def print_answer(query):
    answer = rag_chain.invoke(query)
    print("Resposta:\n", answer, "\n")

In [0]:
query = "O Que é o Microsoft Fabric?"
print_answer(query)

In [0]:
query = "Como Empresas Podem Utilizar o Microsoft Fabric?"
print_answer(query)